# Analisi Prenotazioni Hotel

Dataset: Hotel Booking Demand



In [92]:
import pandas as pd

df = pd.read_csv("Clean Data .csv", sep = ';')

In [95]:
df.head()

,ID,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,arrival_date,stays_in_weekend_nights,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,1,Resort Hotel,0,342,2015,July,27,1,01/07/2015,0,...,No Deposit,NaN,NaN,0,Transient,0,0,0,Check-Out,01/07/2015
1,2,Resort Hotel,0,737,2015,July,27,1,01/07/2015,0,...,No Deposit,NaN,NaN,0,Transient,0,0,0,Check-Out,01/07/2015
2,3,Resort Hotel,0,7,2015,July,27,1,01/07/2015,0,...,No Deposit,NaN,NaN,0,Transient,75,0,0,Check-Out,02/07/2015
3,4,Resort Hotel,0,13,2015,July,27,1,01/07/2015,0,...,No Deposit,304.0,NaN,0,Transient,75,0,0,Check-Out,02/07/2015
4,5,Resort Hotel,0,14,2015,July,27,1,01/07/2015,0,...,No Deposit,240.0,NaN,0,Transient,98,0,1,Check-Out,03/07/2015


# Analisi delle cancellazioni per tipologia di Hotel

In [108]:
cancel_rate = df.groupby('hotel')['is_canceled'].mean() * 100 
print("Percentuale di cancellazione:")
for hotel, rate in cancel_rate.items(): 
    print(f"  {hotel}: {rate:.1f}%") 





Percentuale di cancellazione:
  City Hotel: 41.7%
  Resort Hotel: 27.8%


# Periodo di Maggiore Occupazione in base alla tipologia di Hotel

In [107]:
df_active = df[df['is_canceled'] == 0 ]

month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']

occupancy = df_active.groupby(['hotel', 'arrival_date_month']).size().reset_index(name='prenotazioni')

tabella = occupancy.pivot(index='arrival_date_month', columns='hotel', values='prenotazioni')
tabella = tabella.reindex(month_order)
print("N. Prenotazioni x Mese:")
print(tabella.to_string())

occupancy['arrival_date_month'] = pd.Categorical(
    occupancy['arrival_date_month'], categories=month_order, ordered=True)



print(" Mese con più prenotazioni City Hotel vs Resort Hotel:")
print(f" City Hotel   : {tabella['City Hotel'].idxmax()} ({tabella['City Hotel'].max():,} prenotazioni)")
print(f"Resort Hotel : {tabella['Resort Hotel'].idxmax()} ({tabella['Resort Hotel'].max():,} prenotazioni)")


N. Prenotazioni x Mese:
hotel               City Hotel  Resort Hotel
arrival_date_month                          
January                   2254          1868
February                  3064          2308
March                     4072          2573
April                     4015          2550
May                       4579          2535
June                      4366          2038
July                      4782          3137
August                    5381          3257
September                 4290          2102
October                   4337          2577
November                  2696          1976
December                  2392          2017
 Mese con più prenotazioni City Hotel vs Resort Hotel:
 City Hotel   : August (5,381 prenotazioni)
Resort Hotel : August (3,257 prenotazioni)


# Analisi durata media soggiorno

In [106]:

df['durata_totale'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df_active = df[df['is_canceled'] == 0]


print("Soggiorno Medio")
for hotel in ['City Hotel', 'Resort Hotel']:
    d = df_active[df_active['hotel'] == hotel]['durata_totale']
    print(f"{hotel}:")
    print(f"    Media soggiorno : {d.mean():.1f} notti")


  

Soggiorno Medio
City Hotel:
    Media soggiorno : 2.9 notti
Resort Hotel:
    Media soggiorno : 4.1 notti


# Analisi ADR medio in base al mese e alla tipologia di Hotel 

In [104]:
adr_mese = df_active.groupby(['hotel', 'arrival_date_month'])['adr'].mean().round(2).reset_index()
adr_mese['arrival_date_month'] = pd.Categorical(
    adr_mese['arrival_date_month'], categories=month_order, ordered=True)

adr_mese = adr_mese.sort_values('arrival_date_month')
tabella_adr = adr_mese.pivot(index='arrival_date_month', columns='hotel', values='adr')

tabella_adr = tabella_adr.reindex(month_order)
print("ADR medio per mese e hotel:")
print(tabella_adr.to_string())


print("Mese con tariffa più alta:")
print(f"   City Hotel   : {tabella_adr['City Hotel'].idxmax()} ({tabella_adr['City Hotel'].max():,.2f} €)")
print(f"   Resort Hotel : {tabella_adr['Resort Hotel'].idxmax()} ({tabella_adr['Resort Hotel'].max():,.2f} €)")

ADR medio per mese e hotel:
hotel               City Hotel  Resort Hotel
arrival_date_month                          
January                1880.23        555.94
February               1535.85        775.08
March                  2227.36        917.10
April                  2545.05       1897.25
May                    2476.70       1787.13
June                   2777.20       2566.40
July                   2964.98       4634.73
August                 2645.19       5512.47
September              2785.51       2500.91
October                2873.44       1427.91
November               2185.70        826.72
December               2615.57       2055.15
Mese con tariffa più alta:
   City Hotel   : July (2,964.98 €)
   Resort Hotel : August (5,512.47 €)


# Paese con più prenotazioni per tipologia di Hotel 

In [86]:
country_data = df_active.groupby(['hotel', 'country']).size().reset_index(name='prenotazioni')

top_city = (country_data[country_data['hotel'] == 'City Hotel'].sort_values('prenotazioni', ascending=False).head(10))

top_resort = (country_data[country_data['hotel'] == 'Resort Hotel'].sort_values('prenotazioni',ascending=False).head(10))

print( "Paese con più prenotazioni:")
for hotel, data in [(' City Hotel', top_city), ('  Resort Hotel', top_resort)]:
    primo = data.iloc[0]
    print(f"  {hotel}: {primo['country']} ({primo['prenotazioni']:,} prenotazioni)")
    print ("PRT = Portogallo")



Paese con più prenotazioni:
   City Hotel: PRT (10,879 prenotazioni)
PRT = Portogallo
    Resort Hotel: PRT (10,192 prenotazioni)
PRT = Portogallo
